In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
clean_data = pd.read_csv('sentiment_results.csv')
clean_data.head()

,tweets,lemmatized_tweets,vader_sentiment,textblob_sentiment,vader_compound,textblob_polarity,textblob_subjectivity,sentiment
0,Monkey pox,Monkey pox,-1,-1,0.0000,-0.050000,0.000000,Negative
1,course new scam monkey pox,course new scam monkey pox,-1,1,-0.5719,0.043182,0.227273,Negative
2,Monkeypox virtually avoided less random gay se...,Monkeypox virtually avoided less random gay se...,1,-1,0.1531,-0.083333,0.383333,Positive
3,event monkey pox spread dont sex man man,event monkey pox spread dont sex man man,-1,-1,0.0000,-0.050000,0.000000,Negative
4,dont ask source Ive got bad feeling trying get...,dont ask source Ive got bad feeling trying get...,-1,-1,-0.4588,-0.700000,0.666667,Negative


In [7]:
clean_data.dropna(subset=['tweets'], inplace=True)
clean_data = clean_data[clean_data['tweets'].str.strip() != '']
clean_data

,tweets,lemmatized_tweets,vader_sentiment,textblob_sentiment,vader_compound,textblob_polarity,textblob_subjectivity,sentiment
0,Monkey pox,Monkey pox,-1,-1,0.0000,-0.050000,0.000000,Negative
1,course new scam monkey pox,course new scam monkey pox,-1,1,-0.5719,0.043182,0.227273,Negative
2,Monkeypox virtually avoided less random gay se...,Monkeypox virtually avoided less random gay se...,1,-1,0.1531,-0.083333,0.383333,Positive
3,event monkey pox spread dont sex man man,event monkey pox spread dont sex man man,-1,-1,0.0000,-0.050000,0.000000,Negative
4,dont ask source Ive got bad feeling trying get...,dont ask source Ive got bad feeling trying get...,-1,-1,-0.4588,-0.700000,0.666667,Negative
...,...,...,...,...,...,...,...,...
82886,truly sexual transmission children Monkeypox C...,truly sexual transmission child Monkeypox Chec...,1,1,0.1027,0.500000,0.833333,Positive
82887,vaccines finally available Recently BlaqOut CE...,vaccine finally available Recently BlaqOut CEO...,1,1,0.5994,0.192857,0.571429,Positive
82888,Starting hour pm Sign join conversation hMPXV ...,Starting hour pm Sign join conversation hMPXV ...,1,1,0.6597,0.333333,0.366667,Positive
82889,uptick monkeypox cases Dr Thomas Giordano prov...,uptick monkeypox case Dr Thomas Giordano provi...,1,1,0.6369,1.000000,0.300000,Positive


In [8]:
tfidf_vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1,3))
X_tfid = tfidf_vectorizer.fit_transform(clean_data['lemmatized_tweets'].dropna())
tfidf_df = pd.DataFrame(X_tfid.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

In [9]:
new_df = pd.concat([clean_data.reset_index(drop=True), tfidf_df], axis=1)
new_df

,tweets,lemmatized_tweets,vader_sentiment,textblob_sentiment,vader_compound,textblob_polarity,textblob_subjectivity,sentiment,access,according,...,worried,worry,would,wrong,yeah,year,yes,yet,york,youre
0,Monkey pox,Monkey pox,-1,-1,0.0000,-0.050000,0.000000,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,course new scam monkey pox,course new scam monkey pox,-1,1,-0.5719,0.043182,0.227273,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Monkeypox virtually avoided less random gay se...,Monkeypox virtually avoided less random gay se...,1,-1,0.1531,-0.083333,0.383333,Positive,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,event monkey pox spread dont sex man man,event monkey pox spread dont sex man man,-1,-1,0.0000,-0.050000,0.000000,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,dont ask source Ive got bad feeling trying get...,dont ask source Ive got bad feeling trying get...,-1,-1,-0.4588,-0.700000,0.666667,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82839,truly sexual transmission children Monkeypox C...,truly sexual transmission child Monkeypox Chec...,1,1,0.1027,0.500000,0.833333,Positive,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
82840,vaccines finally available Recently BlaqOut CE...,vaccine finally available Recently BlaqOut CEO...,1,1,0.5994,0.192857,0.571429,Positive,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
82841,Starting hour pm Sign join conversation hMPXV ...,Starting hour pm Sign join conversation hMPXV ...,1,1,0.6597,0.333333,0.366667,Positive,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
82842,uptick monkeypox cases Dr Thomas Giordano prov...,uptick monkeypox case Dr Thomas Giordano provi...,1,1,0.6369,1.000000,0.300000,Positive,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
import gensim
from gensim.models import Word2Vec

tokenized_tweets = new_df['lemmatized_tweets'].dropna().apply(lambda x: x.split())

w2v_model = Word2Vec(sentences=tokenized_tweets, vector_size=100, window=5, min_count=2, workers=4)

def get_w2v_vector(tweet):
    words = tweet.split()
    word_vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    return np.mean(word_vectors, axis=0) if word_vectors else np.zeros(100)

new_df['w2v_vector'] = new_df['lemmatized_tweets'].dropna().apply(lambda x: get_w2v_vector(x))

In [12]:
new_df.head()

,tweets,lemmatized_tweets,vader_sentiment,textblob_sentiment,vader_compound,textblob_polarity,textblob_subjectivity,sentiment,access,according,...,worry,would,wrong,yeah,year,yes,yet,york,youre,w2v_vector
0,Monkey pox,Monkey pox,-1,-1,0.0000,-0.050000,0.000000,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[0.7035701, 0.9362934, -0.28036553, 1.6278687,..."
1,course new scam monkey pox,course new scam monkey pox,-1,1,-0.5719,0.043182,0.227273,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[-0.20491695, 0.56152, -0.44753742, 0.37702236..."
2,Monkeypox virtually avoided less random gay se...,Monkeypox virtually avoided less random gay se...,1,-1,0.1531,-0.083333,0.383333,Positive,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[-0.38586593, -0.086896606, -0.25618204, 0.301..."
3,event monkey pox spread dont sex man man,event monkey pox spread dont sex man man,-1,-1,0.0000,-0.050000,0.000000,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[-0.7764998, 0.056479797, -1.104537, 0.2511763..."
4,dont ask source Ive got bad feeling trying get...,dont ask source Ive got bad feeling trying get...,-1,-1,-0.4588,-0.700000,0.666667,Negative,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"[-0.22420062, 0.49179184, -0.46794012, 0.37647..."


In [ ]:
!pip install transformers datasets torch scikit-learn
